In [1]:
import torch 
from new_dataloader import (BalancedAgeBatchSampler, TestRetestDataset)
from datetime import datetime  
from torch.utils.data import DataLoader
import pandas as pd

class ResBlock(torch.nn.Module):

    def __init__(self,in_channels,out_channels):
        super().__init__()
        self.conv1=torch.nn.Conv3d(in_channels,out_channels,3,1,1)
        self.batchnorm=torch.nn.BatchNorm3d(out_channels,affine=True,track_running_stats=False) # 原 為 False
        self.ELU=torch.nn.ELU()
        self.conv2=torch.nn.Conv3d(out_channels,out_channels,3,1,1)
        self.conv_shortcut=torch.nn.Conv3d(in_channels,out_channels,1,1,0)
        self.maxpooling=torch.nn.MaxPool3d(2,2,ceil_mode=False)
    def forward(self, x):    
        x_=self.conv_shortcut(x)
        x=self.conv1(x)
        x=self.batchnorm(x)
        x=self.ELU(x)
        x=self.conv2(x)
        x=self.batchnorm(x)
        x=self.ELU(x+x_)
        return x
    
class NetResBlock(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.resblock1=ResBlock(1,8)
        self.resblock2=ResBlock(8,16)
        self.resblock3=ResBlock(16,32)
        self.resblock4=ResBlock(32,64)
        self.resblock5=ResBlock(64,128)
        self.resblock6=ResBlock(128,256)
        self.maxpooling1=torch.nn.MaxPool3d(2,2,1,ceil_mode=False)
        self.maxpooling2=torch.nn.MaxPool3d(2,2,(1,0,1),ceil_mode=False)
        
    def forward(self, x):
        x=self.resblock1(x)
        x=self.maxpooling1(x)
        x=self.resblock2(x)
        x=self.maxpooling1(x)
        x=self.resblock3(x)
        x=self.maxpooling2(x)
        x=self.resblock4(x)
        x=self.maxpooling1(x)
        x=self.resblock5(x)
        x=self.maxpooling1(x)
        x=self.resblock6(x)
        x=self.maxpooling2(x)
        return x 
    
class RESNET_MODEL(torch.nn.Module):
    def __init__(self,model_type):
        super().__init__()
        valid_types = {
            "T1",
            "T1_SEX",
            "T1_B0",
            "T1_SEX_B0",
        }
        if model_type not in valid_types:
            raise ValueError(
                f"Unknown model_type: {model_type}. "
                f"Choose from {valid_types}"
            )
        self.model_type = model_type
        self.RES=NetResBlock()
        self.flatten=torch.nn.Flatten()
        self.fc1=torch.nn.Linear(16384,256)
        self.ELU=torch.nn.ELU()
        self.dropout=torch.nn.Dropout(p=0.2)  

        self.fc2_for_no_cat = torch.nn.Linear(256, 1)  # T1-only
        cat_dim = 258 if model_type == "T1_SEX_B0" else 257   # SEX 或 B0：256 + 1 = 257 、 SEX + B0：256 + 2 = 258
        self.fc2_for_cat = torch.nn.Linear(cat_dim,1)
        
    def forward(self,x,gender,scanner):
        
        x=self.RES(x)
        x=self.flatten(x)
        x=self.fc1(x)
        x=self.ELU(x)
        x=self.dropout(x)

        if self.model_type == "T1":
            x = self.fc2_for_no_cat(x)

        elif self.model_type == "T1_SEX":
            x = torch.cat((x, gender),dim=1)
            x = self.fc2_for_cat(x)

        elif self.model_type == "T1_B0":
            x = torch.cat((x, scanner),dim=1)
            x = self.fc2_for_cat(x)

        elif self.model_type == "T1_SEX_B0":
            x = torch.cat((x, gender, scanner),dim=1)
            x = self.fc2_for_cat(x)

        return x

In [8]:
if __name__ == '__main__':

    MODEL_PATHS = {
        "T1": ( r"C:\Lab\models\ResNet\pretrained\model_20250624_095907_881.pt"),
        "T1_SEX": (r"C:\Lab\models\ResNet\pretrained\model_20250705_182002_540.pt"),
        "T1_B0": (r"C:\Lab\models\ResNet\pretrained\model_20250620_095129_948.pt"),
        "T1_SEX_B0": (r"C:\Lab\models\ResNet\pretrained\model_20250628_092154_757.pt")
    }

    RUNS = [
        ("T1", "test"),
        # ("T1_SEX", "test"),
        # ("T1_B0", "test"),
        # ("T1_SEX_B0", "test"),
        # ("T1", "val"),
        # ("T1_SEX", "val"),
        # ("T1_B0", "val"),
        # ("T1_SEX_B0", "val"),
        ("T1", "Maclaren"),
        ("T1", "MMRR"),
        ("T1", "OAS1"),
        # ("T1_SEX", "Maclaren"),
        # ("T1_SEX", "MMRR"),
        # ("T1_SEX", "OAS1"),
        # ("T1_B0", "Maclaren"),
        # ("T1_B0", "MMRR"),
        # ("T1_B0", "OAS1"),
        # ("T1_SEX_B0", "Maclaren"),
        # ("T1_SEX_B0", "MMRR"),
        # ("T1_SEX_B0", "OAS1"),
    ]

    batch_size = 4
    output_dir = r"C:\Lab\models\ResNet\output"
    device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
    print(f"Device     : {device}")
    print(f"Batch size : {batch_size}")

    summary = []
    for model_type, dataset_name in RUNS:

        saved_model_path = MODEL_PATHS[model_type]

        print(f"Model   : {model_type}")
        print(f"Dataset : {dataset_name}")
        print(f"Checkpoint : {saved_model_path}")

        pretrained_model = torch.load(saved_model_path,weights_only=False)  # 載入訓練好的 checkpoint    
        model = RESNET_MODEL(model_type).to(device)  # 建立目前的 ResNet 架構
        model.load_state_dict(pretrained_model.state_dict(),strict=False)  # 將 checkpoint 權重載入目前模型

        # 讀取資料 以及 區分batch
        test_dataset = TestRetestDataset(
            dataset_name=dataset_name
        )

        test_sampler = BalancedAgeBatchSampler(
            test_dataset,
            batch_size=batch_size,
            dataset_type="test",
        )

        test_loader = DataLoader(
            test_dataset,
            batch_sampler=test_sampler,
            pin_memory=True,
        )

        results = []

        # 切換成 evaluation mode，Dropout 會關閉，
        # 但BatchNorm track_running_stats=False，所以 test 時 BatchNorm 仍會使用目前 batch 計算 mean / variance。
        model.eval()

        with torch.no_grad():
            for i, vdata in enumerate(test_loader):

                inputs = vdata['image'].view(-1, 1, 193, 229, 193).to(device)
                labels = vdata['age'].view(-1, 1).to(device)
                gender = (torch.t(vdata['gender'].unsqueeze(0)).int()).to(device)
                scanner = (torch.t(vdata['scanner'].unsqueeze(0)).int()).to(device)

                outputs = model(inputs, gender, scanner)
                preds = outputs.cpu().numpy().flatten()
                trues = labels.cpu().numpy().flatten()

                subjects = vdata["subject_name"]
                image_paths = vdata["image_path"]

                print(
                    f"Batch {i:02d} | "
                    f"N={len(subjects)} | "
                    f"{list(subjects)}"
                )


                # ----------------------------------------------------
                # 儲存每位受試者結果
                # ----------------------------------------------------
                for subject, true_age, pred_age, image_path in zip(subjects, trues, preds, image_paths):

                    results.append({
                        "Dataset": "MMRR",
                        "Subject": subject,
                        "True_Age": float(true_age),
                        "Predicted_Age": float(pred_age),
                        "Abs_Error": abs(float(true_age) - float(pred_age)),
                        "Image_Path": image_path,
                    })

        results_df = pd.DataFrame(results)
        mae = results_df["Abs_Error"].mean()
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        save_path = (
            fr"{output_dir}"
            fr"\ResNet_{dataset_name}_{model_type}_BN_False_{batch_size}_{timestamp}.xlsx"
        )
        results_df.to_excel(save_path, index=False)
        print(f"Results saved to: {save_path}")
        summary.append({
            "Model": model_type,
            "Dataset": dataset_name,
            "N": len(results_df),
            "MAE": mae,
        })

Device     : cuda
Batch size : 4
Model   : T1
Dataset : test
Checkpoint : C:\Lab\models\ResNet\pretrained\model_20250624_095907_881.pt
Batch 00 | N=4 | ['ABIDE2_29314', 'FCON_Beijing_sub00440', 'HCA7453375', 'IXI_626']
Batch 01 | N=4 | ['ABIDE2_29465', 'FCON_Beijing_sub50972', 'FCON_NewYork_sub63915', 'FCON_Munchen_sub26670']
Batch 02 | N=4 | ['FCON_AnnArbor_sub46727', 'OAS1_0136_MR1', 'IXI_040', 'IXI_453']
Batch 03 | N=4 | ['NKI-RS_A00056020', 'FCON_Newark_sub32580', 'ABIDE1_51475', 'HCA7453981']
Batch 04 | N=4 | ['ABIDE2_29353', 'IXI_612', 'NKI-RS_A00083574', 'OAS1_0135_MR1']
Batch 05 | N=4 | ['ABIDE1_50168', 'ABIDE1_50703', 'HCA8174679', 'HCA9126269']
Batch 06 | N=4 | ['ABIDE2_29909', 'ABIDE1_50042', 'FCON_Milwaukee_sub91468', 'HCA6937998']
Batch 07 | N=4 | ['ABIDE2_29472', 'FCON_Berlin_sub23506', 'FCON_Milwaukee_sub56108', 'IXI_560']
Batch 08 | N=4 | ['ABIDE2_28862', 'ABIDE1_50698', 'NKI-RS_A00039353', 'OAS1_0212_MR1']
Batch 09 | N=4 | ['ABIDE1_51257', 'FCON_SaintLouis_sub88150', '

In [9]:
summary_df = pd.DataFrame(summary)

print("\n全部完成")
print(summary_df)


全部完成
  Model   Dataset    N       MAE
0    T1      test  350  3.495995
1    T1  Maclaren  120  8.174681
2    T1      MMRR   42  8.316194
3    T1      OAS1   40  8.167023


In [11]:

from pathlib import Path
import re
import pandas as pd
from datetime import datetime


# ============================================================
# 合併所有 ResNet prediction Excel
# ============================================================

BN = False
BATCH_SIZE = 4

output_dir = Path(r"C:\Lab\models\ResNet\output")

output_file = output_dir / f"ResNet_BN_{BN}_{BATCH_SIZE}.xlsx"


# ------------------------------------------------------------
# 檔名格式：
# ResNet_Maclaren_T1_BN_False_1_20261013_162315.xlsx
#
# 解析成：
# dataset = Maclaren
# model   = T1
# BN      = False
# batch   = 1
# time    = 20261013_162315
# ------------------------------------------------------------

pattern = re.compile(
    r"^ResNet_(Maclaren|MMRR|OAS1|test)_"
    r"(T1_SEX_B0|T1_SEX|T1_B0|T1)_"
    rf"BN_{BN}_{BATCH_SIZE}_"
    r"(\d{8}_\d{6})\.xlsx$"
)


# ------------------------------------------------------------
# 找出每個 Dataset × Model 最新的一份結果
# ------------------------------------------------------------

latest_files = {}

for file_path in output_dir.glob("ResNet_*.xlsx"):

    if file_path.name == output_file.name:
        continue

    match = pattern.match(file_path.name)

    if match is None:
        continue

    dataset_name = match.group(1)
    model_type = match.group(2)
    timestamp_str = match.group(3)

    timestamp = datetime.strptime(
        timestamp_str,
        "%Y%m%d_%H%M%S"
    )

    sheet_name = f"{dataset_name}_{model_type}"

    # 同組合只保留最新結果
    if (
        sheet_name not in latest_files
        or timestamp > latest_files[sheet_name]["timestamp"]
    ):
        latest_files[sheet_name] = {
            "file": file_path,
            "timestamp": timestamp,
            "dataset": dataset_name,
            "model": model_type,
        }


# ------------------------------------------------------------
# 寫入同一個 Excel
# ------------------------------------------------------------

if not latest_files:
    raise FileNotFoundError(
        f"No matching ResNet files in {output_dir}"
    )

with pd.ExcelWriter(
    output_file,
    engine="openpyxl"
) as writer:

    for sheet_name, info in latest_files.items():

        df = pd.read_excel(info["file"])

        result_df = pd.DataFrame({
            "Dataset": info["dataset"],
            "Subject ID": df["Subject"],
            "True Age": df["True_Age"],
            "Predicted Age": df["Predicted_Age"],
        })

        result_df.to_excel(
            writer,
            sheet_name=sheet_name,
            index=False
        )

        print(
            f"{sheet_name:<25s} <- "
            f"{info['file'].name}"
        )


print(f"\n完成：{output_file}")


Maclaren_T1               <- ResNet_Maclaren_T1_BN_False_4_20261014_224138.xlsx
MMRR_T1                   <- ResNet_MMRR_T1_BN_False_4_20261014_224141.xlsx
OAS1_T1                   <- ResNet_OAS1_T1_BN_False_4_20261014_224145.xlsx
test_T1                   <- ResNet_test_T1_BN_False_4_20261014_224129.xlsx

完成：C:\Lab\models\ResNet\output\ResNet_BN_False_4.xlsx
